In [ ]:
# notebooks/evaluation.ipynb (converted to Python for submission)
import pandas as pd
import numpy as np
from typing import List, Dict
import json
from app.retriever.embedding_retriever import SHLRetriever

def calculate_recall_at_k(recommended_urls: List[str], relevant_urls: List[str], k: int) -> float:
    """
    Calculate Recall@K
    """
    if not relevant_urls:
        return 0.0
    
    recommended_at_k = recommended_urls[:k]
    relevant_set = set(relevant_urls)
    hits = sum(1 for url in recommended_at_k if url in relevant_set)
    
    return hits / len(relevant_urls)

def evaluate_model(retriever, train_data_path: str, k: int = 10):
    """
    Evaluate model on training data
    """
    # Load training data
    train_df = pd.read_csv(train_data_path)
    
    recalls = []
    
    for _, row in train_df.iterrows():
        query = row['query']
        relevant_urls = eval(row['relevant_urls']) if isinstance(row['relevant_urls'], str) else row['relevant_urls']
        
        # Get recommendations
        results = retriever.retrieve(query, k=k*2)
        recommended_urls = [r[0]['url'] for r in results]
        
        # Calculate recall
        recall = calculate_recall_at_k(recommended_urls, relevant_urls, k)
        recalls.append(recall)
        
        print(f"Query: {query[:50]}...")
        print(f"Recall@{k}: {recall:.3f}")
        print()
    
    mean_recall = np.mean(recalls)
    print(f"Mean Recall@{k}: {mean_recall:.3f}")
    
    return mean_recall

def optimize_parameters(train_data_path: str):
    """
    Experiment with different parameters to optimize performance
    """
    from sentence_transformers import SentenceTransformer
    
    # Test different models
    models = [
        "all-MiniLM-L6-v2",  # Fast, good performance
        "all-mpnet-base-v2",  # Slower but better
        "multi-qa-mpnet-base-dot-v1",  # Optimized for QA
    ]
    
    results = {}
    
    for model_name in models:
        print(f"\nTesting model: {model_name}")
        retriever = SHLRetriever(model_name=model_name)
        
        # Load assessments (assuming they're already crawled)
        with open("data/raw/shl_catalog.json", 'r') as f:
            assessments = json.load(f)
        
        retriever.build_index(assessments)
        
        # Evaluate
        mean_recall = evaluate_model(retriever, train_data_path)
        results[model_name] = mean_recall
    
    # Print results
    print("\n=== Model Comparison ===")
    for model, recall in sorted(results.items(), key=lambda x: x[1], reverse=True):
        print(f"{model}: Recall@10 = {recall:.3f}")
    
    return results

# Run evaluation
if __name__ == "__main__":
    # Initialize retriever
    retriever = SHLRetriever()
    retriever.load("data/processed/index")
    
    # Evaluate on training data
    mean_recall = evaluate_model(retriever, "data/evaluation/train.csv", k=10)
    
    # Generate predictions for test set
    test_df = pd.read_csv("data/evaluation/test.csv")
    
    predictions = []
    for query in test_df['query']:
        results = retriever.hybrid_retrieve(query, k=10)
        for assessment, score in results:
            predictions.append({
                'query': query,
                'Assessment_url': assessment['url']
            })
    
    # Save predictions
    pred_df = pd.DataFrame(predictions)
    pred_df.to_csv('submission.csv', index=False)
    print(f"Saved {len(pred_df)} predictions to submission.csv")